# 🕌 Indexing Pipeline for Online Efteely


# 1. Setup


In [ ]:
import torch
import pandas as pd
import shutil
import os
from tqdm.notebook import tqdm
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from google.colab import files

print(f"CUDA Available: {torch.cuda.is_available()})")

# 2. Load Dataset


In [ ]:
DATA_PATH = "collected.csv"  
CHROMA_DIR = "chroma_db"

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

print("⏳ Loading dataset...")
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded {len(df)} fatwas.")

# 3. Prepare Documents & Metadata

In [ ]:

print("📝 Preparing documents and metadata...")
texts, metadatas = [], []
for _, row in df.iterrows():
    text = f"Question: {row.get('question','')}\nAnswer: {row.get('answer','')}"
    url = str(row.get("URL") or row.get("url") or row.get("link") or row.get("source") or "").strip()
    texts.append(text)
    metadatas.append({"source": url, "link": url})

print(f"✅ Prepared {len(texts)} documents.")

# 4. Initialize Multilingual E5 Model (GPU)


In [ ]:
print("🚀 Loading E5 Model on GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": device},
    encode_kwargs={"batch_size": 128, "normalize_embeddings": True}
)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
    collection_name="langchain"
)

# 5. Start Indexing (Batch Processing)


In [ ]:
BATCH_SIZE = 1000
print(f"🏗️ Starting Indexing...")

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Indexing Progress"):
    batch_texts = texts[i : i + BATCH_SIZE]
    batch_metas = metadatas[i : i + BATCH_SIZE]
    vectorstore.add_texts(texts=batch_texts, metadatas=batch_metas)

print(f"🎉 Completed! Total in DB: {vectorstore._collection.count()}")

# 6. Verification & Download


In [ ]:
print("📦 Zipping Database...")
shutil.make_archive("chroma_db_final", 'zip', CHROMA_DIR)

print("✅ Download starting...")
files.download("chroma_db_final.zip")